# Visual CNN: train, evaluate and export

Status: NOT RUN. Requires permitted images and batch-separated train/val/test labels. MobileNetV3 Small is a starting architecture, not a validated final selection. GPU recommended; CPU is slower. Begin with batch size 16 and reduce on memory errors. Training duration depends on data and hardware. Softmax is uncalibrated. No prediction certifies food safety.

In [ ]:
from pathlib import Path
import os, sys
REPO_ROOT = Path(os.environ.get("SB_REPO_ROOT", "/CHANGE/TO/UPLOADED/REPOSITORY"))
assert (REPO_ROOT / "requirements-training.txt").is_file(), "Set REPO_ROOT to the folder containing README.md"
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# Run this only in your selected Colab/Kaggle/local training environment.
# Kaggle requires internet enabled to install/download pretrained weights.
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-training.txt"])

In [ ]:
from build.imagery.cnn.data import validate_manifest
DATA_ROOT = Path("/CHANGE/TO/DATA")
MANIFEST = DATA_ROOT / "manifest.json"
m = validate_manifest(MANIFEST, DATA_ROOT)
from collections import Counter
print(Counter((s["split"],s["label"]) for s in m["samples"]))

Inspect a sample of each class and split before training. Confirm batch groups never cross splits. Detection and multitask heads require additional labels and are not enabled.

In [ ]:
from build.imagery.cnn.pipeline import train
EPOCHS = 10
RESUME = None  # e.g. REPO_ROOT / "output/checkpoints/visual_last.pt"
report = train(MANIFEST, DATA_ROOT, epochs=EPOCHS, batch_size=16, seed=42, resume=RESUME)
report

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(REPO_ROOT / "output/visualizations/visual_training.png")))
from build.imagery.cnn.pipeline import saliency
sample = next(s for s in m["samples"] if s["split"]=="test")
saliency(REPO_ROOT/"output/models/model_1_visual_cnn", DATA_ROOT/sample["path"], REPO_ROOT/"output/visualizations/saliency.png")
display(Image(filename=str(REPO_ROOT/"output/visualizations/saliency.png")))

Review per-class precision/recall, confusion matrix and coverage before model use. Test thresholds and calibration on validation data in a follow-up experiment. Copy output/models/model_1_visual_cnn, checkpoints, logs and visualizations to persistent storage before the notebook session ends.

## Download the complete model
Run only after successful training. The ZIP includes weights, metadata and evaluation. Save checkpoints separately if you want to resume later.

In [ ]:
from scripts.export_model_bundle import export
bundle = export('model_1_visual_cnn')
from IPython.display import display, FileLink
display(FileLink(str(bundle.relative_to(REPO_ROOT))))
print("Download this file using the notebook file browser if the link is unavailable:", bundle)
